# Moment 2: transport_budget_share

This is the fragile one. The plan's own definitional-trap note lists three published transport-share
figures differing fourfold, none of which is quite `c`: a search cost paid by the unemployed, not
a consumption share or a cost borne by people who already have the job. NHTS 2020 turns out to
have a real, purpose-built question block for exactly this population -- but no codebook shipped
with the raw extracted CSVs, and the obvious cost variable is a red herring. Every non-obvious
choice below is documented because a future reader (including the marker) needs to be able to
follow the chain, not just trust the final number.

## Finding the right variables

`Q46Lookwork` ("Have looked for a job") and `Q47Travlooking` ("Travelled looking for job") --
labels confirmed against the DataFirst release's own `.dta` file (`nhts-2020-v1`), not guessed --
identify 4,655 of 145,385 respondents who travelled while looking for work. Before that label
file was available, the same population was reconstructed from the raw CSV's skip pattern alone:
`Q46`'s "No" count (15,225) exactly equals `Q47`'s "Not applicable" count, and `Q46`'s "Yes" count
(8,319) exactly equals `Q47`'s "Yes" (4,655) plus "No" (3,664) -- internally consistent, and the
labelled file confirmed the reconstruction was right.

The obvious cost variable, `Q434bCost`, turns out to be labelled "Vehicle cost to work place" --
a private-vehicle-specific field, mostly Not-applicable for this subsample (only 143 of 4,655 have
a real value) because most people who travel while job-hunting don't drive their own car. The
right variables are `Q429b3Cost1` through `Q432b3Cost4` -- "Cost of Nth mode of travel to
destination," one field per mode of travel actually used (up to four), specifically on the `b`
question block that matches `Q47Travlooking`'s own numbering, distinct from an earlier,
similarly-named block (`Q4293Cost1` etc.) that asks about the *regular commute to an existing job*,
not job-search travel. Getting this pairing wrong would have quietly measured commuting costs
for employed people instead of search costs for job seekers -- the exact substitution the plan's
definitional-trap note warns against.

## Both sides of the ratio are conditional and daily -- neither claims a search frequency

`Q429b3Cost1`..`Q432b3Cost4` are costs for a single reported travel day, conditional on that
person having actually travelled while job-hunting (`Q47Travlooking == "Yes"`). The natural wage
comparator, `Q410Salary`, is reported monthly. Dividing the day-level cost by the monthly wage
directly compares two different time units and understates the ratio by roughly a factor of 22 --
an early pass did exactly that and got 0.0049, an order of magnitude below every other published
estimate in the plan's own trap note, which was the signal something was wrong before the number
was trusted any further. The fix converts the wage to a daily rate (South Africa's conventional
~21.7 working days per month) so both sides of the ratio share the same (daily) time unit.

**This is a unit conversion, not a frequency claim.** Converting a monthly wage to `wage / 21.7`
says only "here is what one day of this wage is worth" -- it does not assert, and the moment does
not need, that job-search travel happens on 21.7 days a month, or on any particular number of
days a month. The numerator is already conditional on a travel day having happened at all (that's
what `Q47Travlooking == "Yes"` selects for); the denominator is a same-unit conversion of the wage,
not a frequency assumption smuggled into the denominator. `src/moments.py`'s
`transport_budget_share()` was rewritten to match this exactly: it conditions on an actual search
trip happening and compares that trip's cost to one day's wage, saying nothing about how many such
trips occur in a month either. Neither side of the comparison makes a monthly-frequency claim, and
the earlier framing in this notebook ("costs recur at roughly a daily rate") suggested one where
none exists -- corrected here. See DECISIONS.md, "transport_budget_share and its NHTS target were
measuring two different things."

## The wage benchmark

`Q410Salary` ("Total Salary/Pay from main job"), restricted to the ~20,867 respondents with a real
value and a stated pay period (`Q411Payperiod`: per month, per week x4.345, or annually /12),
weighted, gives a median monthly wage of R4,300 (2020 terms) -- median chosen over the weighted
mean (R7,754) because a right-skewed income distribution makes the mean a poor stand-in for the
single representative wage the model's `WAGE=1.0` numeraire is meant to describe. This wage comes
from NHTS 2020 itself, not a more recent QLFS earnings figure, specifically so both sides of the
ratio share one survey and one year -- mixing a 2020 transport cost with a 2025/26 wage would
introduce an inflation-adjustment problem this notebook doesn't need to solve. The whole moment is
therefore in **2020 Rand terms**, a real limitation carried into the Calibration chapter, not
smoothed over.

## Uncertainty

The ratio combines two independently-estimated quantities (mean cost over 4,655 travellers, median
wage over 20,867 earners), so the standard error is bootstrapped (2,000 resamples, weighted,
resampling both populations independently each draw) rather than derived from a closed-form
formula that would need to assume away the covariance between them.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Set THESIS_DATA_ROOT to wherever you extracted the DataFirst downloads -- see data/README.md.
# No committed absolute path here: this notebook must run unmodified on a machine that isn't
# this thesis's own.
DATA_ROOT = Path(os.environ["THESIS_DATA_ROOT"])
PERSON_PATH = (
    DATA_ROOT / "nhts2020-v1" / "nhts-2020-v1" / "nhts-2020-v1-stata" / "nhts-2020-person.dta"
)

COST_COLS = ["Q429b3Cost1", "Q430b3Cost2", "Q431b3Cost3", "Q432b3Cost4"]
WORKING_DAYS_PER_MONTH = 21.7  # South Africa's conventional figure, used only to make the
# day-level transport cost and the monthly wage comparable -- see the markdown cell above.

labelled = pd.read_stata(
    PERSON_PATH,
    columns=["Q46Lookwork", "Q47Travlooking", "Weight", "Q411Payperiod", *COST_COLS],
    convert_categoricals=True,
)
raw = pd.read_stata(PERSON_PATH, columns=[*COST_COLS, "Q410Salary"], convert_categoricals=False)

print(f"total respondents: {len(labelled):,}")
print(labelled["Q47Travlooking"].value_counts(dropna=False))

In [ ]:
travellers = labelled[labelled["Q47Travlooking"] == "Yes"].copy()
travellers_raw = raw.loc[travellers.index]

total_cost = pd.Series(0.0, index=travellers.index)
for col in COST_COLS:
    is_real = ~travellers[col].isin(["Unspecified", "Not applicable"])
    total_cost += travellers_raw[col].where(is_real, 0.0)

cost_vals = total_cost.to_numpy()
cost_weights = travellers["Weight"].to_numpy()

print(f"job-search travellers: {len(travellers):,}")
print(pd.Series(cost_vals).describe())
mean_daily_cost = np.average(cost_vals, weights=cost_weights)
print(f"\nweighted mean daily job-search transport cost: R{mean_daily_cost:.2f}")

In [ ]:
wage_mask = (
    (raw["Q410Salary"] > 0)
    & (raw["Q410Salary"] < 999_000)  # excludes the 9,999,999 / -999 / -998 sentinel codes
    & labelled["Q411Payperiod"].isin(["Per month", "Per week", "Annually"])
)
wage_period = labelled.loc[wage_mask, "Q411Payperiod"]
period_factor = wage_period.map({"Per month": 1.0, "Per week": 4.345, "Annually": 1 / 12})
monthly_wage_vals = (raw.loc[wage_mask, "Q410Salary"] * period_factor).to_numpy()
wage_weights = labelled.loc[wage_mask, "Weight"].to_numpy()

print(f"respondents with a usable wage figure: {wage_mask.sum():,}")
print(pd.Series(monthly_wage_vals).describe())

order = np.argsort(monthly_wage_vals)
cum_w = np.cumsum(wage_weights[order])
median_monthly_wage = monthly_wage_vals[order][cum_w >= cum_w[-1] / 2][0]
mean_monthly_wage = np.average(monthly_wage_vals, weights=wage_weights)
print(f"\nweighted median monthly wage: R{median_monthly_wage:.2f}")
print(f"weighted mean monthly wage (right-skewed, not the denominator): R{mean_monthly_wage:.2f}")

In [ ]:
daily_wage = median_monthly_wage / WORKING_DAYS_PER_MONTH
point_estimate = mean_daily_cost / daily_wage
print(f"daily wage equivalent: R{daily_wage:.2f}")
print(
    f"transport_budget_share (point estimate): {point_estimate:.4f} ({point_estimate * 100:.2f}%)"
)

rng = np.random.default_rng(42)
n_boot = 2000
boot_shares = np.empty(n_boot)
cost_p = cost_weights / cost_weights.sum()
wage_p = wage_weights / wage_weights.sum()
for i in range(n_boot):
    ci = rng.choice(len(cost_vals), size=len(cost_vals), replace=True, p=cost_p)
    wi = rng.choice(len(monthly_wage_vals), size=len(monthly_wage_vals), replace=True, p=wage_p)
    boot_cost = cost_vals[ci].mean()
    boot_wage = np.median(monthly_wage_vals[wi]) / WORKING_DAYS_PER_MONTH
    boot_shares[i] = boot_cost / boot_wage

bootstrap_se = boot_shares.std(ddof=1)
ci_lo, ci_hi = np.percentile(boot_shares, [2.5, 97.5])
print(f"\nbootstrap SE: {bootstrap_se:.4f}")
print(f"95% bootstrap CI: [{ci_lo:.4f}, {ci_hi:.4f}]")

In [ ]:
moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "transport_budget_share"
moments.loc[row, "value"] = round(float(point_estimate), 4)
moments.loc[row, "standard_error"] = round(float(bootstrap_se), 4)
moments.loc[row, "period"] = "2020"
moments.loc[row, "source"] = (
    "Statistics South Africa. National Household Travel Survey 2020 [dataset]. "
    "Cape Town: DataFirst [distributor]. Job-search travel cost (Q429b3Cost1-Q432b3Cost4, "
    "conditional on Q47Travlooking=Yes) as a share of median monthly wage (Q410Salary), "
    "both person-weighted, wage converted to a daily rate at 21.7 working days/month. "
    "2020 Rand terms -- see notebook 02 for the full methodology and its limitations."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]